# Fine-tuning SegFormer для сегментации автомобилей

**Цель:** обучить модель **SegFormer** с **менее 1 млн. обучаемых параметров** для задачи semantic segmentation и добиться качества **IoU ≥ 0.92** на тестовой выборке. Размер изображений для обучений и теста должен быть **<= 384х384**. (см пункт "4. Проверки")

Для выполнения используется **уникальный сплит** датасета [Carvana Image Masking Challenge](https://www.kaggle.com/c/carvana-image-masking-challenge)

Работаем **только с частью `train`** датасета
## Система оценки

### Подход к решению (макс 5 баллов)

*Промежуточные оценки выставляются по широте экспериментов*

**1 балл**

- обучена одна модель SegFormer
- стандартный pipeline
- стандартные transforms / loss / optimizer

---

**3 балла**

- рассмотрено **минимум 2 решения**

например:

- разные модели (`SegFormer-B0`, `SegFormer-B1`) и/или веса
- или разная стратегия finetuning (заморозка encoder целиком / частично / добавление слоев)

или

- проведено **2–3 эксперимента оптимизации**

например:

- scheduler
- аугментации
- разные функции потерь

---

**5 баллов**

- минимум **2 архитектурных решения**
- **+ минимум 2 эксперимента оптимизации**

например:

- BCE vs Dice loss
- разные стратегии resize / crop
- scheduler
- аугментации

---

### Качество решения vs cтоимость (макс 5 баллов)

Метрика: **Intersection over Union (IoU)**

**+1 балл**

если

```
IoU ≥ 0.87
```

**+2 балла**

если

```
IoU ≥ 0.92
```

**+2 балла**:

уложились с качеством в **10 эпох** обучения

---

**Можно получить дополнительный +1 балл в качестве поощрения от ревьюера**

за:
- вспомогательную визуалиацию, поиск новых закономерностей
- оригинальные архитектурные решения

## 1. Формирование уникального сплита

In [20]:
import os
import numpy as np
import pandas as pd
import torch
import random
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import albumentations as A


def set_seed_to_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)


DATE_OF_BIRTH: int = # ВСТАВИТЬ ДЕНЬ РОЖДЕНИЯ
MONTH_OF_BIRTH: int = # ВСТАВИТЬ МЕСЯЦ РОЖДЕНИЯ
HEIGHT = # ВСТАВИТЬ РОСТ
LASTNAME = # ВСТАВИТЬ ФАМИЛИЮ

seed = (DATE_OF_BIRTH + MONTH_OF_BIRTH) * (ord(LASTNAME[0])%(HEIGHT%10))

set_seed_to_all(seed)

In [17]:
from sklearn.model_selection import train_test_split


images = sorted(os.listdir("carvana-image-masking-challenge/train/"))
df = pd.DataFrame({"image": images})
df["car_id"] = df["image"].apply(lambda x: x.split("_")[0])
car_ids = df["car_id"].unique()
subset_cars = random.sample(list(car_ids), 100)
my_data = df[df["car_id"].isin(subset_cars)]

train_ids, temp_ids = train_test_split(
    subset_cars,
    test_size=0.3,
    random_state=seed
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.5,
    random_state=seed
)

train_df = df[df["car_id"].isin(train_ids)]
val_df = df[df["car_id"].isin(val_ids)]
test_df = df[df["car_id"].isin(test_ids)]

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 1120
Val: 240
Test: 240


In [18]:
len(my_data)

1600

## 2. Подготовка данных
Реализовать самим

In [ ]:
class CarvanaDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        image_dir: str,
        mask_dir: str,
        transform: transforms.Compose | A.Compose | None = None
    ):
        self.df = df
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # YOUR CODE HERE
        pass

## 3. Подбор архитектуры и обучение
Самим подобрать Segformer (и версию, и претрейн веса), решить какую часть обучать, настроить гиперпараметры, выбрать лосс и обучить сеть  
Рекомендую брать из трансформерс и искать хороший претрейн  
```
from transformers import SegformerForSemanticSegmentation
```

## 4. Проверки

In [21]:
def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    device: str | torch.device
) -> float:
    """
    Подсчёт среднего IoU по датасету.
    """
    # РЕАЛИЗОВАТЬ САМИМ
    pass

In [ ]:
# проверка на обучаемые параметры
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Trainable params:", trainable_params)
assert trainable_params < 1_000_000

# провкрка на размеры
for images, masks in test_loader:
    assert images[0].shape[0] <= 384 and images[0].shape[1] <= 384


# проверка качества
iou = evaluate(model, test_loader)

if iou > 0.85:
    print("Добились IoU > 0.87")

if iou > 0.90:
    print("Добились IoU > 0.92")

print("IoU:", iou)